In [ ]:
import pandas as pd
import numpy as np

import sys
sys.path.append('..')
from helpers import get_county2zone

In [2]:
state_abbrev_name_map = {
    'Alabama': 'AL',
    'Alaska': 'AK',
    'Arizona': 'AZ',
    'Arkansas': 'AR',
    'California': 'CA',
    'Colorado': 'CO',
    'Connecticut': 'CT',
    'District of Columbia': 'DC',
    'Delaware': 'DE',
    'Florida': 'FL',
    'Georgia': 'GA',
    'Hawaii': 'HI',
    'Idaho': 'ID',
    'Illinois': 'IL',
    'Indiana': 'IN',
    'Iowa': 'IA',
    'Kansas': 'KS',
    'Kentucky': 'KY',
    'Louisiana': 'LA',
    'Maine': 'ME',
    'Maryland': 'MD',
    'Massachusetts': 'MA',
    'Michigan': 'MI',
    'Minnesota': 'MN',
    'Mississippi': 'MS',
    'Missouri': 'MO',
    'Montana': 'MT',
    'Nebraska': 'NE',
    'Nevada': 'NV',
    'New Hampshire': 'NH',
    'New Jersey': 'NJ',
    'New Mexico': 'NM',
    'New York': 'NY',
    'North Carolina': 'NC',
    'North Dakota': 'ND',
    'Ohio': 'OH',
    'Oklahoma': 'OK',
    'Oregon': 'OR',
    'Pennsylvania': 'PA',
    'Rhode Island': 'RI',
    'South Carolina': 'SC',
    'South Dakota': 'SD',
    'Tennessee': 'TN',
    'Texas': 'TX',
    'Utah': 'UT',
    'Vermont': 'VT',
    'Virginia': 'VA',
    'Washington': 'WA',
    'West Virginia': 'WV',
    'Wisconsin': 'WI',
    'Wyoming': 'WY'
}

In [66]:
def get_distpv_generation(year):
    df = pd.read_excel(f'../data/distpv_generation/epa_03_21_{year}.xlsx').loc[1:].reset_index(drop=True)
    df.loc[0] = (
        (df.loc[0].ffill().str.lower().str.replace(' sector', '_') + df.loc[3].ffill().str.lower().str.replace('year ', ''))
    )
    df.columns = pd.MultiIndex.from_arrays([df.loc[2].ffill(), df.loc[0].ffill()])
    df = df.loc[4:].set_index(df.columns[0])
    df = df['Estimated Small Scale Generation'].dropna(how='all')
    df = df.reset_index(names=['state'])
    df['state'] = (
        df['state']
        .astype(str)
        .str.replace('(', '')
        .str.replace(')', '')
        .str.replace('\'', '')
        .str.replace(',', '')
        .map(state_abbrev_name_map)
        .dropna()
    )
    df = (
        df.dropna(subset='state')
        .set_index('state')
        [[f'residential_{year}', f'commercial_{year}', f'industrial_{year}']]
        * 1e3
    )
    df.columns = ['residential_mwh', 'commercial_mwh', 'industrial_mwh']
    df['non_residential_mwh'] = df['commercial_mwh'] + df['industrial_mwh']

    df = df[['residential_mwh', 'non_residential_mwh']]

    return df

In [67]:
def get_distpv_energy_sold_back(year):
    df = pd.read_excel(f'../data/sales/Net_Metering_{year}.xlsx')
    
    if year > 2016:
        df.columns = [np.nan if col.startswith('Unnamed') else col for col in df.columns]
        df.loc[-1] = df.columns
        df = df.sort_index().reset_index(drop=True)
    
    df.columns = pd.MultiIndex.from_arrays([df.loc[0].ffill(), df.loc[1].ffill(), df.loc[2].ffill()])
    df = df.loc[3:]
    df = df.set_index(('Utility Characteristics', np.nan, 'State'))
    df.index = df.index.rename('state')
    df = df['Photovoltaic']['Energy Sold Back MWh']
    df.columns.names = ['']
    
    for col in df.columns:
        df[f"{col.lower()}_mwh"] = pd.to_numeric(df[col], errors='coerce')
        df = df.drop(columns=col)
    
    df = df.reset_index()
    df = (
        df.groupby('state')
        .sum(numeric_only=True)
    )

    df['non_residential_mwh'] = df['commercial_mwh'] + df['industrial_mwh']

    df = df[['residential_mwh', 'non_residential_mwh']]
    
    return df

In [76]:
annual_distpv_consumption_by_year = {}
for year in range(2016, 2024):
    df_generation = get_distpv_generation(year)
    df_energy_sold_back = get_distpv_energy_sold_back(year)

    df_consumption = df_generation.sub(df_energy_sold_back)
    df_consumption = df_consumption.loc[df_consumption.index.isin(county2zone.state)]
    annual_distpv_consumption_by_year[year] = df_consumption

C:\Users\cobika\AppData\Local\Temp\1\ipykernel_29952\4186576468.py:10: PerformanceWarning: indexing past lexsort depth may impact performance.
  df = df.loc[4:].set_index(df.columns[0])
C:\Users\cobika\AppData\Local\Temp\1\ipykernel_29952\4186576468.py:10: PerformanceWarning: indexing past lexsort depth may impact performance.
  df = df.loc[4:].set_index(df.columns[0])
C:\Users\cobika\AppData\Local\Temp\1\ipykernel_29952\4186576468.py:10: PerformanceWarning: indexing past lexsort depth may impact performance.
  df = df.loc[4:].set_index(df.columns[0])
C:\Users\cobika\AppData\Local\Temp\1\ipykernel_29952\4186576468.py:10: PerformanceWarning: indexing past lexsort depth may impact performance.
  df = df.loc[4:].set_index(df.columns[0])
C:\Users\cobika\AppData\Local\Temp\1\ipykernel_29952\4186576468.py:10: PerformanceWarning: indexing past lexsort depth may impact performance.
  df = df.loc[4:].set_index(df.columns[0])
C:\Users\cobika\AppData\Local\Temp\1\ipykernel_29952\4186576468.py:10:

In [ ]:
county2zone = get_county2zone(2023)
distpv_generation_by_year_state = distpv_generation_by_year_state.loc[(
    distpv_generation_by_year_state.index.get_level_values('state').isin(county2zone.state)
)]
distpv_energy_sold_back_by_year_state = distpv_energy_sold_back_by_year_state.loc[(
    distpv_energy_sold_back_by_year_state.index.get_level_values('state').isin(county2zone.state)
)]

In [9]:
distpv_consumption_by_year_state = (
    distpv_generation_by_year_state.sub(distpv_energy_sold_back_by_year_state)
)

In [23]:
distpv_cap = (
    pd.read_csv("../data/BTM PV - county.csv")
    .set_index('r')
    .drop(columns='State')
    .T
)
distpv_cap.index = distpv_cap.index.astype(int)
distpv_cap = distpv_cap.loc[range(2016, 2024)]

In [25]:
county_distpv_generation_profiles_by_sector = {}
for sector in ['residential', 'commercial']:
    df_cf = pd.read_hdf(f'../data/distpv_profiles/processed/county_rooftop_pv_cf_{sector}.h5')
    df_cf = df_cf.loc[df_cf.index.year.isin(range(2016, 2024))].copy()
    df_cf = df_cf.set_index(df_cf.index.year, append=True)

    df_gen = df_cf.mul(distpv_cap, level=1)
    county_distpv_generation_profiles_by_sector[sector] = df_gen.droplevel(1)

In [138]:
county_distpv_generation_proportions_by_sector = {}
for sector in ['residential', 'commercial']:
    distpv_generation_proportions = county_distpv_generation_profiles_by_sector[sector].copy()
    distpv_generation_proportions = (
        distpv_generation_proportions.groupby(distpv_generation_proportions.index.year)
        .sum()
        .transpose()
        .merge(county2zone.set_index('FIPS')[['state']], left_index=True, right_index=True)
    )
    for load_year in range(2016, 2024):
        distpv_generation_proportions[load_year] = (
            distpv_generation_proportions[load_year] /
            distpv_generation_proportions.groupby('state')[load_year].transform('sum')
        )
    county_distpv_generation_proportions_by_sector[sector] = (
        distpv_generation_proportions.set_index('state', append=True)
        .bfill(axis=1)
    )
    county_distpv_generation_proportions_by_sector[sector].index.names = ['FIPS', 'state']

In [139]:
residential_consumption = pd.concat({k: v['residential_mwh'] for k,v in annual_distpv_consumption_by_year.items()}, axis=1)
non_residential_consumption = pd.concat({k: v['non_residential_mwh'] for k,v in annual_distpv_consumption_by_year.items()}, axis=1)

In [142]:
county_distpv_residential_consumption = (
    county_distpv_generation_proportions_by_sector['residential']
    .mul(residential_consumption, level=1)
    .droplevel('state')
)
county_distpv_non_residential_consumption = (
    county_distpv_generation_proportions_by_sector['commercial']
    .mul(non_residential_consumption, level=1)
    .droplevel('state')
)

In [154]:
county_distpv_residential_consumption.to_csv('../data/county_distpv_residential_consumption.csv')
county_distpv_non_residential_consumption.to_csv('../data/county_distpv_non_residential_consumption.csv')